In [618]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

In [619]:
# Load the raw JSON
data_path = Path("../data/raw_credit_applications.json")  # or the correct relative/path in your repo
with open(data_path, "r") as f:
    df = json.load(f)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    str    
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     str    
 3   applicant_info.full_name          502 non-null    str    
 4   applicant_info.email              502 non-null    str    
 5   applicant_info.ssn                497 non-null    str    
 6   applicant_info.ip_address         497 non-null    str    
 7   applicant_info.gender             501 non-null    str    
 8   applicant_info.date_of_birth      501 non-null    str    
 9   applicant_info.zip_code           501 non-null    str    
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financials.debt_to_

In [620]:
# Check for duplicate application IDs
total_rows = df.shape[0]
unique_ids = df["_id"].nunique()
duplicate_rows = total_rows - unique_ids

print("Total rows:", total_rows)
print("Unique _id values:", unique_ids)
print("Duplicate rows (by _id):", duplicate_rows)

before = df.shape[0]
df = df.drop_duplicates(subset="_id", keep="first")
after = df.shape[0]

df['_temp_nan_count'] = df.isna().sum(axis=1)
df = (df.groupby('_id', group_keys=False)
      .apply(lambda g: g.loc[g['_temp_nan_count'].idxmin()])
      .drop('_temp_nan_count', axis=1))

after = df.shape[0]
print("Rows before dropping duplicates:", before)
print("Rows after (keeping row with LEAST NaNs):", after)
print("Rows removed:", before - after)

Total rows: 502
Unique _id values: 500
Duplicate rows (by _id): 2
Rows before dropping duplicates: 502
Rows after (keeping row with LEAST NaNs): 500
Rows removed: 2


In [621]:
# combine salary and income into one column
# done prior to drop salary in next step before cleaning data
print("Pre-merge: income NaN:", df['financials.annual_income'].isnull().sum())
df['financials.annual_income'] = df['financials.annual_income'].fillna(df['financials.annual_salary'])

print("Post-merge income NaN:", df['financials.annual_income'].isnull().sum())

Pre-merge: income NaN: 5
Post-merge income NaN: 0


In [622]:
# 1. Overview of missing data
def missing_summary(df):
    missing_counts = df.isna().sum().sort_values(ascending=False)
    missing_percent = (missing_counts / len(df) * 100).round(1)
    return pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": missing_percent
    })

ms = missing_summary(df)
print(ms)

                                  missing_count  missing_percent
notes                                       500            100.0
financials.annual_salary                    495             99.0
loan_purpose                                450             90.0
processing_timestamp                        438             87.6
decision.rejection_reason                   292             58.4
decision.approved_amount                    208             41.6
decision.interest_rate                      208             41.6
applicant_info.ssn                            4              0.8
applicant_info.ip_address                     4              0.8
financials.savings_balance                    0              0.0
decision.loan_approved                        0              0.0
spending_behavior                             0              0.0
financials.debt_to_income                     0              0.0
financials.annual_income                      0              0.0
applicant_info.zip_code  

In [623]:
# Drop columns with >50% missing values
threshold = 50
cols_to_drop = ms[ms["missing_percent"] > threshold].index.tolist()
print("Dropping columns with >50% missing:", cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("\nRemaining columns:", df.shape[1])

Dropping columns with >50% missing: ['notes', 'financials.annual_salary', 'loan_purpose', 'processing_timestamp', 'decision.rejection_reason']
Remaining columns: 15


In [624]:
# ensure numeric columns are numeric
numeric_cols = [
    "financials.annual_income",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.interest_rate",
    "decision.approved_amount",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)

financials.annual_income            float64
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.interest_rate              float64
decision.approved_amount            float64
dtype: object


In [625]:
def check_numerical_for_limits(df, variable, lb, ub=None):
    """Check bounds [lb, ub) and replace out-of-bounds with NaN"""
    out_of_bounds = (df[variable] < lb)
    if ub is not None:
        out_of_bounds |= (df[variable] > ub)
    count_invalid = out_of_bounds.sum()
    pct_invalid = (count_invalid / len(df)) * 100
    print(f"{variable} out-of-bounds [{lb}, {ub or 'inf'}): {count_invalid} ({pct_invalid:.1f}%)")
    df.loc[out_of_bounds, variable] = np.nan
    return df

# handle Datime consistency and transforming to age
today = pd.to_datetime('2026-03-05')
df['applicant_info.date_of_birth'] = pd.to_datetime(df['applicant_info.date_of_birth'], dayfirst=False, yearfirst=False, errors='coerce')
df['applicant_info.age'] = (today - df['applicant_info.date_of_birth']).dt.days // 365

# transform gender for consistency
df['applicant_info.gender'] = df['applicant_info.gender'].str.strip().str.upper()
df['applicant_info.gender'] = df['applicant_info.gender'].map({
    'M': 'Male',
    'MALE': 'Male',
    'F': 'Female', 
    'FEMALE': 'Female',
    '': np.nan
}).ffill()

# unify email adresses and check for consistency
df['applicant_info.email'] = df['applicant_info.email'].str.lower().str.strip()
invalid_email = ~df['applicant_info.email'].str.contains(r'@[^@]+\.[^@]+$', na=False)
print(f"Invalid emails (no @domain.tld): {invalid_email.sum()} ({invalid_email.mean()*100:.1f}%)")
df.loc[invalid_email, 'applicant_info.email'] = np.nan

# check names for consistency
single_word_names = df['applicant_info.full_name'].str.split().str.len() == 1
no_capital = ~df['applicant_info.full_name'].str[0].str.isupper()
print(f"Single-word names: {single_word_names.sum()} ({single_word_names.mean()*100:.1f}%)")
print(f"No capital start: {no_capital.sum()} ({no_capital.mean()*100:.1f}%)")

# transform ssn patterns
ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'
invalid_ssn = ~df['applicant_info.ssn'].str.match(ssn_pattern, na=False)
print(f"Invalid SSN format (not XXX-XX-XXXX): {invalid_ssn.sum()} ({invalid_ssn.mean()*100:.1f}%)")
df.loc[invalid_ssn, 'applicant_info.ssn'] = np.nan

# ZIP: Enforce 5 digits (truncate extra)
zip_str = df['applicant_info.zip_code'].astype(str)
long_zip = zip_str.str.len() > 5
print(f"ZIP >5 digits: {long_zip.sum()}")
df.loc[long_zip, 'applicant_info.zip_code'] = zip_str.loc[long_zip].str[:5]
df['applicant_info.zip_code'] = pd.to_numeric(df['applicant_info.zip_code'], errors='coerce').astype('Int64')

# check for non-boolean loan_approved
print("Non-boolean count loan_approved:", (~df['decision.loan_approved'].isin([True, False])).sum())

# Calculate absolute debt = income * debt_to_income
df['applicant_info.debt'] = df['financials.annual_income'] * df['financials.debt_to_income']

# flag and check numerical variables for upper and lower bounds
df = check_numerical_for_limits(df, "financials.annual_income", 0)
df = check_numerical_for_limits(df, "financials.savings_balance", 0)
df = check_numerical_for_limits(df, "financials.debt_to_income", 0, 1)
df = check_numerical_for_limits(df, "financials.credit_history_months", 0)
df = check_numerical_for_limits(df, "decision.approved_amount", 0)
df = check_numerical_for_limits(df, "decision.interest_rate", 0)
df = check_numerical_for_limits(df, "applicant_info.age", 0, 100)

df

Invalid emails (no @domain.tld): 10 (2.0%)
Single-word names: 0 (0.0%)
No capital start: 0 (0.0%)
Invalid SSN format (not XXX-XX-XXXX): 4 (0.8%)
ZIP >5 digits: 0
Non-boolean count loan_approved: 0
financials.annual_income out-of-bounds [0, inf): 0 (0.0%)
financials.savings_balance out-of-bounds [0, inf): 1 (0.2%)
financials.debt_to_income out-of-bounds [0, 1): 1 (0.2%)
financials.credit_history_months out-of-bounds [0, inf): 2 (0.4%)
decision.approved_amount out-of-bounds [0, inf): 0 (0.0%)
decision.interest_rate out-of-bounds [0, inf): 0 (0.0%)
applicant_info.age out-of-bounds [0, 100): 0 (0.0%)


,spending_behavior,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.interest_rate,decision.approved_amount,applicant_info.age,applicant_info.debt
_id,,,,,,,,,,,,,,,,,
app_001,"[{'category': 'Fitness', 'amount': 576}]",Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000.0,37.0,0.42,0.0,False,NaN,NaN,39.0,42840.0
app_002,"[{'category': 'Education', 'amount': 533}]",Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,Male,1999-08-01,10020,41000.0,5.0,0.36,18200.0,False,NaN,NaN,26.0,14760.0
app_003,"[{'category': 'Healthcare', 'amount': 450}]",Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,Female,1982-08-24,90213,65000.0,74.0,0.43,7090.0,True,3.4,76000.0,43.0,27950.0
app_004,"[{'category': 'Transportation', 'amount': 329}...",Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,Female,NaT,90217,69000.0,9.0,0.41,10327.0,False,NaN,NaN,NaN,28290.0
app_005,"[{'category': 'Insurance', 'amount': 585}]",Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,Female,NaT,90296,39000.0,76.0,0.06,15011.0,False,NaN,NaN,NaN,2340.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
app_496,"[{'category': 'Fitness', 'amount': 175}]",Rachel Miller,rachel.miller69@protonmail.com,953-69-6408,10.194.95.93,Female,1979-11-12,90211,87000.0,40.0,0.31,23949.0,True,6.2,25000.0,46.0,26970.0
app_497,"[{'category': 'Groceries', 'amount': 423}]",Patrick Wilson,patrick.wilson77@hotmail.com,655-58-3025,10.131.43.18,Male,NaT,90233,48000.0,4.0,0.10,21540.0,False,NaN,NaN,NaN,4800.0
app_498,"[{'category': 'Education', 'amount': 177}, {'c...",James Rivera,james.rivera25@gmail.com,942-34-6834,172.18.221.237,Male,1989-10-19,90276,86000.0,33.0,0.29,36901.0,False,NaN,NaN,36.0,24940.0


In [626]:
path = Path("../data/clean_credit_applications.csv")
df.to_csv(path, index=False)
print("Shape:", df.shape)

Shape: (500, 17)
